# Transformers e ensembles — pipeline cache-aware

O notebook pode executar toda a pipeline. Se ela já tiver sido rodada no cluster, as chamadas obrigatoriamente recuperam os caches correspondentes e não repetem o treinamento. A busca usa validação interna; o ranking final usa `val_f1_macro` no `dataset_validation`. Todos os treinos têm limite de 100 épocas, early stopping e restauração do melhor checkpoint.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay
here = Path.cwd().resolve()
BACKEND = here if (here / 'machine_learning').exists() else (here / 'backend' if (here / 'backend').exists() else here.parent)
sys.path.insert(0, str(BACKEND))
from machine_learning.cache import ModelCache
from machine_learning.transformer.runner import (
    TRANSFORMER_EMBEDDINGS, run_transformer_finalist, finalize_transformer_pipeline
)
# False: usa o cache e calcula apenas o que estiver faltando.
# True: refaz tudo; para isso, prefira o job SLURM.
FORCE_RETRAIN = False
for embedding in TRANSFORMER_EMBEDDINGS:
    run_transformer_finalist(embedding, force_retrain=FORCE_RETRAIN)
summary = finalize_transformer_pipeline()
print('Protocolo:', summary['protocol_version'], '| campeão:', summary['winner'])

## Busca em três etapas por embedding

In [ ]:
for finalist in summary['finalists']:
    embedding = finalist['embedding']
    search_path = BACKEND / 'experiment_results' / 'transformer' / 'search' / embedding / 'summary.json'
    search = json.loads(search_path.read_text(encoding='utf-8'))
    print(f'\n### {embedding.upper()}')
    for stage in search['stages']:
        rows = [{**candidate['config'],
                 'internal_val_f1_macro': candidate['internal_val_f1_macro'],
                 'internal_val_pr_auc': candidate['internal_val_pr_auc'],
                 'internal_val_recall_scam': candidate['internal_val_recall_scam']}
                for candidate in stage['candidates']]
        frame = pd.DataFrame(rows).sort_values('internal_val_f1_macro', ascending=False)
        print(stage['name'], '— vencedor')
        display(frame.head(10))

## Early stopping dos vencedores (seed visual fixa 42)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, finalist in zip(axes, summary['finalists']):
    seed42 = next(row for row in finalist['seeds'] if row['seed'] == 42)
    history = ModelCache.load_history('transformer', seed42['run_id'])
    ax.plot(history['epoch'], history['train_f1'], label='treino')
    ax.plot(history['epoch'], history['val_f1'], label='validação interna')
    ax.set_title(f"{finalist['embedding']} — {len(history)}/100 épocas")
    ax.set_xlabel('época'); ax.set_ylabel('F1 macro'); ax.legend()
plt.tight_layout(); plt.show()

## Três seeds: teste interno e dataset_validation

In [ ]:
finalists = pd.DataFrame([{
    'embedding': row['embedding'], 'configuração': row['configuration'],
    'test_f1_macro_mean': row['test_f1_macro_mean'], 'test_f1_macro_std': row['test_f1_macro_std'],
    'val_f1_macro_mean': row['val_f1_macro_mean'], 'val_f1_macro_std': row['val_f1_macro_std'],
} for row in summary['finalists']])
display(finalists.style.format({c: '{:.4f}' for c in finalists.columns if 'f1_' in c}).highlight_max(subset=['val_f1_macro_mean']))

## Majority voting, soft voting e soma de logits calibrados

In [ ]:
display(pd.DataFrame(summary['ensembles'])[['name', 'f1_macro', 'pr_auc', 'recall_scam', 'roc_auc']].sort_values('f1_macro', ascending=False))
print('Temperaturas ajustadas somente na validação interna:', summary['temperatures'])

## Diagnósticos finais no dataset_validation

In [ ]:
items = []
for finalist in summary['finalists']:
    seed42 = next(row for row in finalist['seeds'] if row['seed'] == 42)
    items.append((finalist['embedding'], ModelCache.load_prediction_bundle('transformer', seed42['run_id'], 'validation')))
for ensemble in summary['ensembles']:
    items.append((ensemble['name'], ModelCache.load_prediction_bundle('transformer', 'ensemble', ensemble['name'])))
fig, axes = plt.subplots(3, len(items), figsize=(5 * len(items), 12))
for col, (name, bundle) in enumerate(items):
    y, pred, prob = bundle['y_true'], bundle['y_pred'], bundle['y_prob']
    ConfusionMatrixDisplay.from_predictions(y, pred, ax=axes[0, col], colorbar=False)
    RocCurveDisplay.from_predictions(y, prob, ax=axes[1, col])
    PrecisionRecallDisplay.from_predictions(y, prob, ax=axes[2, col])
    axes[0, col].set_title(name)
plt.tight_layout(); plt.show()

## Ranking final por val_f1_macro

In [ ]:
ranking = pd.DataFrame(summary['ranking'])
display(ranking.style.format({'val_f1_macro': '{:.4f}', 'std': '{:.4f}'}).highlight_max(subset=['val_f1_macro']))
print('Vencedor:', summary['winner'])